# Classification

**CubicNova Data Science Roadmap — Module 06**

This notebook demonstrates binary and multiclass classification, baseline comparison, model evaluation, and probability thresholds.

> Run cells from top to bottom. The examples use datasets bundled with Scikit-learn or generated locally.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    precision_recall_curve, ConfusionMatrixDisplay
)

## 2. Multiclass classification with Iris

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
print("Feature matrix:", X.shape)
print("Features:", iris.feature_names)
print("Target classes:", iris.target_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

## 3. Train and evaluate Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=iris.target_names, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred, display_labels=iris.target_names, cmap=None
)
plt.title("Iris: confusion matrix")
plt.tight_layout()
plt.show()

## 4. Compare against a baseline and decision tree

In [ ]:
models = {
    "Most-frequent baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
}
rows = []
for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    pred_i = estimator.predict(X_test)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred_i),
        "macro_f1": f1_score(y_test, pred_i, average="macro", zero_division=0),
    })
display(pd.DataFrame(rows).set_index("model").round(3))

## 5. Binary classification with a reproducible synthetic dataset

In [ ]:
X_bin, y_bin = make_classification(
    n_samples=1000, n_features=20, n_informative=8, n_redundant=4,
    weights=[0.85, 0.15], random_state=42
)
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_bin, y_bin, test_size=0.20, random_state=42, stratify=y_bin
)

binary_model = RandomForestClassifier(n_estimators=200, random_state=42)
binary_model.fit(X_train_bin, y_train_bin)
pred_bin = binary_model.predict(X_test_bin)
prob_bin = binary_model.predict_proba(X_test_bin)[:, 1]

## 6. Evaluate binary classification

In [ ]:
print("Accuracy:", round(accuracy_score(y_test_bin, pred_bin), 3))
print("Precision:", round(precision_score(y_test_bin, pred_bin, zero_division=0), 3))
print("Recall:", round(recall_score(y_test_bin, pred_bin, zero_division=0), 3))
print("F1:", round(f1_score(y_test_bin, pred_bin, zero_division=0), 3))
print("ROC-AUC:", round(roc_auc_score(y_test_bin, prob_bin), 3))
print("Confusion matrix:\n", confusion_matrix(y_test_bin, pred_bin))
print(classification_report(y_test_bin, pred_bin, zero_division=0))

## 7. Inspect precision and recall at different thresholds

Use validation data to choose a threshold in a real workflow. This test-set illustration is for understanding how predictions change only; do not use the final test set to tune a production threshold.

In [ ]:
thresholds_to_compare = [0.30, 0.50, 0.70]
threshold_rows = []
for threshold in thresholds_to_compare:
    threshold_pred = (prob_bin >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_test_bin, threshold_pred, zero_division=0),
        "recall": recall_score(y_test_bin, threshold_pred, zero_division=0),
        "f1": f1_score(y_test_bin, threshold_pred, zero_division=0),
    })
display(pd.DataFrame(threshold_rows).set_index("threshold").round(3))

## 8. Precision-recall curve

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test_bin, prob_bin)
plt.figure(figsize=(6, 4))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-recall curve (illustrative test evaluation)")
plt.tight_layout()
plt.show()

## 9. Key takeaways

- Classification predicts categories, while regression predicts numeric values.
- Accuracy should not be the only metric when class imbalance or error costs matter.
- Precision focuses on predicted positives; recall focuses on actual positives.
- A confusion matrix makes error types visible.
- Compare with a baseline and evaluate models on held-out data.
- Tune models and thresholds using training/validation data; reserve the test set for final evaluation.
- Consider class imbalance, calibration, data leakage, and deployment context.

## Practice

Complete `practice.md` and compare your answers with `solution.md`.

## Further reading

- [Scikit-learn classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)
- [Scikit-learn metrics API](https://scikit-learn.org/stable/api/sklearn.metrics.html)